# LegalQA v9.2 — grouped-dev candidate oracle và selector calibration
Attach source v9.2, official data, bốn approved base models, completed global dense index và QLoRA `split` artifact. Notebook dùng `selector_train.json` cho kNN nên 400 reference dev không lọt vào train-memory. Agentic được gate và chấm trên candidate-specific evidence.

In [ ]:
TRIALS = 3000
WORKING = '/kaggle/working/legalqa_selector_calibration'
AUTO_RESUME = True  # Attach đúng một selector_recovery_latest.zip để tiếp tục session trước


In [ ]:
from pathlib import Path
import os, shutil, subprocess, sys
input_root = Path('/kaggle/input')
def is_v9_source(path):
    return (path / 'kaggle' / 'run_selector_calibration.py').is_file() and (path / 'src' / 'legalqa' / 'train_memory.py').is_file()
source_projects = sorted({p.parent for p in input_root.rglob('pyproject.toml') if is_v9_source(p.parent)})
if not source_projects:
    raise RuntimeError('Không tìm thấy source LegalQA phù hợp')
def source_score(path):
    project_meta = (path / 'pyproject.toml').read_text(encoding='utf-8', errors='ignore')
    pipeline = (path / 'src' / 'legalqa' / 'pipeline.py').read_text(encoding='utf-8', errors='ignore')
    return (100 if 'version = \"0.9.2\"' in project_meta else 0) + (50 if 'agentic-v3' in pipeline else 0) + len(list((path / 'wheelhouse').glob('*.whl')))
selected_source = max(source_projects, key=lambda path: (source_score(path), str(path)))
if source_score(selected_source) < 162:
    raise RuntimeError(f'Source chưa đủ bản vá v9.2/wheelhouse: {[(str(p), source_score(p)) for p in source_projects]}')
project = Path('/kaggle/working/legalqa_source')
if project.exists():
    shutil.rmtree(project)
shutil.copytree(selected_source, project)
wheels = sorted(str(path) for path in (project / 'wheelhouse').glob('*.whl'))
subprocess.run([sys.executable, '-m', 'pip', 'install', '--no-index', '--no-deps', '--upgrade', *wheels], check=True)
os.environ['PYTHONPATH'] = str(project / 'src')
os.environ['PYTHONUTF8'] = '1'
os.environ['HF_HUB_OFFLINE'] = '1'
os.environ['TRANSFORMERS_OFFLINE'] = '1'
os.environ['TOKENIZERS_PARALLELISM'] = 'false'
subprocess.run(['nvidia-smi', '--query-gpu=index,name,memory.total', '--format=csv'], check=True)


In [ ]:
recovery_candidates = sorted(input_root.rglob('selector_recovery_latest.zip')) if AUTO_RESUME else []
if len(recovery_candidates) > 1:
    raise RuntimeError(f'Có nhiều selector recovery ZIP: {recovery_candidates}')
command = [
    sys.executable, str(project / 'kaggle' / 'run_selector_calibration.py'),
    '--input-root', '/kaggle/input',
    '--working', WORKING,
    '--trials', str(TRIALS),
]
if recovery_candidates:
    command.extend(['--resume-bundle', str(recovery_candidates[0])])
print(' '.join(command))
subprocess.run(command, check=True)


In [ ]:
import json
root = Path(WORKING)
summary = json.loads((root / 'calibration_summary.json').read_text(encoding='utf-8'))
assert 300 <= summary['grouped_dev_count'] <= 500
assert (root / 'tuned_selection.yaml').is_file()
print(json.dumps(summary, ensure_ascii=False, indent=2))
print('Lưu tuned_selection.yaml thành Kaggle Dataset và attach vào notebook public v9.')
